# Temporal Feature Selection

This notebook prepares the engineered temporal dataset for model training.

The temporal exploratory analysis suggested that congestion signals may propagate across operational cycles. Based on those observations, lagged versions of key operational indicators were engineered in the previous stage.

The objective of this notebook is to:

1. Load the engineered temporal dataset
2. Define the target variable and predictor features
3. Verify dataset consistency
4. Ensure the selected predictors remain appropriate given the dataset size

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/JNPA_feature_engineered_temporal.csv")

df.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,...,Total_TEUs_Handled,Date,Rail_Friction,Is_Monsoon,Stress,Risk,Volume_Lag1,Parking_Dwell_Lag1,Rail_Friction_Lag1,Stress_Lag1
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,...,467614,2023-02-01,2.900000,0,0,1,522592.0,1.88,4.741758,0.0
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,...,560076,2023-03-01,2.856502,0,0,0,467614.0,2.55,2.900000,0.0
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,...,503668,2023-04-01,2.094488,0,0,1,560076.0,3.65,2.856502,0.0
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,...,538247,2023-05-01,2.965116,0,0,0,503668.0,4.00,2.094488,0.0
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,...,523948,2023-06-01,2.782609,1,523948,0,538247.0,2.33,2.965116,0.0


## Target Variable Definition

The congestion risk indicator used in this study was defined during the feature engineering stage.

A binary risk flag was created based on the 75th percentile of import dwell time. This threshold acts as an early congestion indicator rather than waiting for the demurrage threshold of 72 hours to be reached.

Risk = 1 represents months where dwell time exceeds the congestion threshold.

In [2]:
df["Risk"].value_counts()

Risk
0    25
1     9
Name: count, dtype: int64

## Predictor Variables

Based on the temporal exploratory analysis, the following lagged operational indicators are selected as predictors:

Volume_Lag1 – previous-month container throughput pressure

Parking_Dwell_Lag1 – previous-month gate congestion indicator

Rail_Friction_Lag1 – previous-month rail evacuation imbalance

Stress_Lag1 – previous-month seasonal operational pressure

In [3]:
features = ["Volume_Lag1","Parking_Dwell_Lag1","Rail_Friction_Lag1","Stress_Lag1"]

target = "Risk"

X = df[features]
y = df[target]

print("Selected Features:", features)

Selected Features: ['Volume_Lag1', 'Parking_Dwell_Lag1', 'Rail_Friction_Lag1', 'Stress_Lag1']


## Event Per Variable (EPV) Consideration

The dataset contains a limited number of congestion events due to the monthly aggregation level.

To avoid model overfitting in logistic regression, the number of predictor variables must remain small relative to the number of positive events.

Although classical guidelines recommend approximately 10 events per predictor variable, modern studies have shown that slightly lower ratios may still produce stable estimates when regularization or cross-validation techniques are used.

Given the small dataset size, the model will be validated using Leave-One-Out Cross Validation (LOOCV) to maximize training data usage.

In [4]:
event_count = y.sum()

print("Number of congestion events:", event_count)
print("Number of predictors:", len(features))

Number of congestion events: 9
Number of predictors: 4


## Final Dataset for Temporal Model

The feature matrix and target vector prepared in this notebook will be used in the next stage for training and evaluating the temporal congestion prediction model.